In [36]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
from clase_reportes import ReportClass
rc = ReportClass()
ruta = None
if ruta:
    ruta = Path(ruta)
    ruta_kits = ruta / 'data' / 'kits.xlsx'
    df_kits = pd.read_excel(ruta_kits)
else:
    ruta = rc.validar_ruta()
    ruta_kits = ruta / 'data' / 'kits.xlsx'
    df_kits = pd.read_excel(ruta_kits)


ruta_base =ruta / 'file' / 'BASE VENTAS 2025.xlsx'

# Cargar el archivo de Excel
df_ventas = pd.read_excel(ruta_base)  # Reemplaza con la ruta de tu archivo

# Verifica si falta incluir kit en el archivo kits.xlsx
df_explosion_prueba = pd.merge(df_ventas, df_kits, left_on="PRODUCTO", right_on="KIT", indicator=True, how='left')

productos_con_kit = [
    producto for producto in df_explosion_prueba[df_explosion_prueba['_merge']=='left_only']['PRODUCTO_x'].unique()
    if 'KIT' in str(producto)
]
print(f"Productos con 'KIT' sin correspondencia en df_kits: {productos_con_kit}")

# Unir df_ventas con df_kits para explotar los kits
df_explosion = pd.merge(df_ventas, df_kits, left_on="PRODUCTO", right_on="KIT")

# Agregar una columna para indicar el origen (kit o individual)
df_explosion["ORIGEN"] = "KIT"

# Calcular las cantidades de productos
df_explosion["CANTIDAD_PRODUCTO"] = df_explosion["CANTIDAD"]

# Calcular el valor por producto en los kits
# df_explosion["VALOR_POR_PRODUCTO"] = df_explosion["TOTAL($)"] / df_explosion.groupby("KIT")["PRODUCTO_x"].transform("count")
conteo_facturas = df_explosion.groupby(['PRODUCTO_x','NUMERO_FACTURA'])['NUMERO_FACTURA'].transform('count')
df_explosion["VALOR_POR_PRODUCTO"] = df_explosion['TOTAL($)'] / conteo_facturas



# Agrupar y sumar las cantidades de productos de kits
df_resultado_kits = df_explosion.groupby(['CATEGORÍA', "PRODUCTO_y", "MES", "ORIGEN", ])["CANTIDAD_PRODUCTO"].sum().reset_index()
df_resultado_kits.columns = ['CATEGORÍA',"PRODUCTO", "MES","ORIGEN", "CANTIDAD_TOTAL"]

# # Agrupar por categorias
# df_resultado_kits_cat = df_explosion.groupby(["PRODUCTO_y", "MES", "ORIGEN",'CATEGORÍA'])["CANTIDAD_PRODUCTO"].sum().reset_index()
# df_resultado_kits.columns = ["PRODUCTO", "MES","ORIGEN",'CATEGORÍA', "CANTIDAD_TOTAL"]


# Filtrar productos individuales
df_ventas_individuales = df_ventas[~df_ventas["PRODUCTO"].str.startswith(("[PCNKIT","[TNGKIT","[B8KIT"))].reset_index(drop=True)
df_ventas_individuales["ORIGEN"] = "INDIVIDUAL"



# Seleccionar y renombrar columnas para que coincidan con df_resultado_kits
df_ventas_individuales = df_ventas_individuales[['CATEGORÍA', "PRODUCTO", "MES","ORIGEN" , "CANTIDAD", "TOTAL($)"]]
df_ventas_individuales.columns = ['CATEGORÍA',"PRODUCTO", "MES", "ORIGEN", "CANTIDAD_TOTAL", "INGRESO_TOTAL"]
 

# Combinar los resultados de kits y productos individuales
df_final = pd.concat([df_resultado_kits, df_ventas_individuales], ignore_index=True)


## Categoria producto ingresos

df_ingresos_kits_cate = df_explosion.groupby(["CATEGORÍA","PRODUCTO_y", "MES", ]).agg({
    "CANTIDAD_PRODUCTO": "sum",
    "VALOR_POR_PRODUCTO": "sum"
}).reset_index()
df_ingresos_kits_cate.columns = ["CATEGORÍA","PRODUCTO", "MES",  "CANTIDAD_TOTAL", "INGRESO_TOTAL"]
df_ingresos_cate= pd.concat([df_ingresos_kits_cate, df_ventas_individuales], ignore_index=True)


## ingresos

# Agrupar y sumar las cantidades de productos de kits
df_ingresos_kits = df_explosion.groupby(["PRODUCTO_y", "MES"]).agg({
    "CANTIDAD_PRODUCTO": "sum",
    "VALOR_POR_PRODUCTO": "sum"
}).reset_index()
df_ingresos_kits.columns = ["PRODUCTO", "MES", "CANTIDAD_TOTAL", "INGRESO_TOTAL"]
df_ingresos= pd.concat([df_ingresos_kits, df_ventas_individuales], ignore_index=True)


# Lista con el orden correcto de los meses en español
orden_meses = [
    'ENERO', 'FEBRERO', 'MARZO', 'ABRIL', 'MAYO', 'JUNIO',
    'JULIO', 'AGOSTO', 'SEPTIEMBRE', 'OCTUBRE', 'NOVIEMBRE', 'DICIEMBRE'
]

# Convertir la columna MES a tipo categoría con orden explícito
df_final['MES'] = pd.Categorical(df_final['MES'], categories=orden_meses, ordered=True)

# Ahora ordenar por MES
df_final = df_final.sort_values(by='MES')

# Categorias pivot table
pivot_table_por_mes_categoria = df_final.pivot_table(
    index=[ "CATEGORÍA","PRODUCTO",],  # Filas: productos
    columns="MES",     # Columnas: meses
    values="CANTIDAD_TOTAL",  # Valores: cantidades
    aggfunc="sum",     # Función de agregación: suma
    fill_value=0,
    observed=True 
            # Rellenar valores faltantes con 0
).reset_index()



# Crear la pivot table
pivot_table_por_mes = df_final.pivot_table(
    index="PRODUCTO",  # Filas: productos
    columns="MES",     # Columnas: meses
    values="CANTIDAD_TOTAL",  # Valores: cantidades
    aggfunc="sum",     # Función de agregación: suma
    fill_value=0,
    observed=True 
            # Rellenar valores faltantes con 0
).reset_index()


# Crear la pivot table
pivot_table_mes_origen = df_final.pivot_table(
    index="PRODUCTO",  # Filas: productos
    columns=["MES", "ORIGEN"],  # Columnas: meses y origen (kit o individual)
    values="CANTIDAD_TOTAL",  # Valores: cantidades
    aggfunc="sum",     # Función de agregación: suma
    fill_value=0,
    observed=True      # Rellenar valores faltantes con 0
).reset_index()


# Crear la pivot table con el nuevo formato
pivot_table_resumida = df_final.pivot_table(
    index=["ORIGEN", "PRODUCTO", ],  # Filas: Producto y tipo (Kit o Individual)
    columns="MES",                 # Columnas: Meses
    values="CANTIDAD_TOTAL",        # Valores: Cantidad total
    aggfunc="sum",                  # Sumar cantidades
    fill_value=0,
    observed=True                    # Reemplazar NaN con 0
).reset_index()


# Convertir la columna MES a tipo categoría con orden explícito
df_ingresos['MES'] = pd.Categorical(df_ingresos['MES'], categories=orden_meses, ordered=True)

# Ahora ordenar por MES
df_ingresos = df_ingresos.sort_values(by='MES')


# Crear la pivot table para las cantidades
pivot_ingresos_cantidades = df_ingresos.pivot_table(
    index="PRODUCTO",  # Filas: productos
    columns="MES",     # Columnas: meses
    values="CANTIDAD_TOTAL",  # Valores: cantidades
    aggfunc="sum",     # Función de agregación: suma
    fill_value=0,
    observed=True       # Rellenar valores faltantes con 0
).reset_index()

# Crear la pivot table para los ingresos
pivot_table_ingresos = df_ingresos.pivot_table(
    index="PRODUCTO",  # Filas: productos
    columns="MES",     # Columnas: meses
    values="INGRESO_TOTAL",  # Valores: ingresos
    aggfunc="sum",     # Función de agregación: suma
    fill_value=0, 
    observed=True       # Rellenar valores faltantes con 0
).reset_index()


# Crear la pivot table categorias
pivot_table_ingresos_cate = df_ingresos.pivot_table(
    index=["CATEGORÍA","PRODUCTO", ],  # Filas: productos
    columns="MES",     # Columnas: meses
    values="INGRESO_TOTAL",  # Valores: cantidades
    aggfunc="sum",     # Función de agregación: suma
    fill_value=0,
    observed=True        # Rellenar valores faltantes con 0
).reset_index()

ruta_file = ruta / 'file' 
cant_ing_cate = pivot_table_por_mes_categoria.melt(
    id_vars=["CATEGORÍA","PRODUCTO", ],
    var_name="MES", 
    value_name="CANTIDAD_TOTAL"
)
ingre_cate=pivot_table_ingresos_cate.melt(
id_vars=["CATEGORÍA","PRODUCTO"],
var_name="MES",
value_name="INGRESO_TOTAL"
)
df_categorias= cant_ing_cate.merge(ingre_cate, on=["PRODUCTO", "CATEGORÍA", "MES"], how="left")


# Diccionario de meses en español
meses_es = {
    'enero': 1, 'febrero': 2, 'marzo': 3,
    'abril': 4, 'mayo': 5, 'junio': 6,
    'julio': 7, 'agosto': 8, 'septiembre': 9,
    'octubre': 10, 'noviembre': 11, 'diciembre': 12
}

# Convertir los nombres de mes en mayúsculas a minúsculas antes de mapear
df_categorias['Mes_num'] = df_categorias['MES'].str.lower().map(meses_es)

# Crear columna de fecha con el día 1 y año 2025
df_categorias['Fecha'] = pd.to_datetime({
    'year': 2025,
    'month': df_categorias['Mes_num'],
    'day': 1
})


Productos con 'KIT' sin correspondencia en df_kits: ['[PCN15] KIT HYDRO CLEAN HC']


In [38]:

ruta_file.mkdir(parents=True, exist_ok=True)  # Crear la carpeta si no existe
print(f"Carpeta '{ruta_file}' creada o ya existe.")
# Guardar la pivot table en un archivo de Excel
pivot_table_por_mes.to_excel(ruta_file / 'pivot_table_por_mes.xlsx', index=False) ## Es igual a "pivot_table_cantidades_por_mes.xlsx"
# Guardar la pivot table en un archivo de Excel
pivot_table_mes_origen.to_excel(ruta_file / "pivot_table_por_mes_y_origen.xlsx")
# Guardar la pivot table en un archivo de Excel
pivot_table_resumida.to_excel(ruta_file / "pivot_table_resumida.xlsx", index=False)

pivot_ingresos_cantidades.to_excel(ruta_file /"pivot_table_cantidades_por_mes.xlsx", index=False)

pivot_table_ingresos.to_excel(ruta_file / "pivot_table_ingresos_por_mes.xlsx", index=False)

df_categorias.to_excel(ruta_file / "categorias_df.xlsx", index=False)


Carpeta 'C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\file' creada o ya existe.
